# KOTFlash — полный прогон на Tesla T4

Этот ноутбук создаёт отдельное Python 3.12.16-окружение с pip, устанавливает совместимый стек PyTorch 2.6.0 + CUDA 12.4 + Triton 3.2.x, клонирует актуальную ветку main и запускает correctness suite вместе с attention, decode и synthetic decoder-block benchmarks.

**Перед запуском:** в Colab выбери `Runtime → Change runtime type → T4 GPU`. Скрипт намеренно остановится, если обнаружит не Tesla T4 / SM75 или несовместимую версию библиотек. Результаты — это attention/decoder-block benchmarks, а не обещание ускорения полной LLM.

Опциональная последняя ячейка отдельно собирает CUDA C++ append extension. Она требует доступного `nvcc` и может быть пропущена.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, **kwargs)

# Install uv into the current Colab runtime, then create an isolated Python 3.12.16 environment.
run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
UV = shutil.which("uv")
if UV is None:
    candidate = Path(sys.prefix) / "bin" / "uv"
    if candidate.is_file():
        UV = str(candidate)
if UV is None:
    raise RuntimeError("uv was installed but its executable was not found on PATH")

run([UV, "python", "install", "3.12.16"])
ENV_DIR = Path("/content/kotflash-env")
run([UV, "venv", "--seed", "--python", "3.12.16", ENV_DIR])
PY = str(ENV_DIR / "bin" / "python")
# Fail here with a clear message rather than halfway through package installation.
run([PY, "-m", "pip", "--version"])
print("Isolated interpreter:", PY)


In [ ]:
# Clone the merged main branch, not the old feature branch.
import subprocess
import shutil
from pathlib import Path

REPO = Path("/content/KOTFlash")
BRANCH = "main"
if REPO.exists() and not (REPO / ".git").exists():
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/AitiKot/KOTFlash.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

subprocess.run([PY, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([
    PY, "-m", "pip", "install", "--index-url",
    "https://download.pytorch.org/whl/cu124", "torch==2.6.0"
], check=True)
subprocess.run([PY, "-m", "pip", "install", "numpy", "pytest", "build", "ninja"], check=True)
subprocess.run([PY, "-m", "pip", "install", "-e", ".[t4,test]"], cwd=REPO, check=True)
print("Installed KOTFlash from:", REPO)


In [ ]:
# Verify the interpreter and GPU before spending time on the full benchmark suite.
import subprocess

check = r'''import torch, triton, kotflash
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA unavailable")
print("Capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
print("KOTFlash:", kotflash.__file__)
assert torch.cuda.is_available(), "Enable a GPU runtime in Colab"
assert torch.cuda.get_device_capability(0) == (7, 5), "This notebook is for SM75 / Tesla T4"
assert "T4" in torch.cuda.get_device_name(0), "Refusing to label a non-T4 GPU run as T4 validation"
assert torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4"
assert triton.__version__.split(".")[:2] == ["3", "2"]
'''
subprocess.run([PY, "-c", check], check=True)


In [ ]:
# Full physical-T4 validation: compileall, the full test suite, and attention/decode/block benchmarks.
# The script writes JSON and pytest logs into /content/KOTFlash/artifacts/.
import os
import subprocess

env = os.environ.copy()
env["PATH"] = str(Path(PY).parent) + os.pathsep + env.get("PATH", "")
subprocess.run(["bash", "benchmarks/run_t4_validation.sh"], cwd=REPO, env=env, check=True)
print("Validation artifacts:", sorted(str(p) for p in (REPO / "artifacts").glob("t4-validation-*")))


In [ ]:
# Optional: build and test the first-party CUDA C++ K/V append extension.
# Skip this cell when nvcc is unavailable; the standard Triton validation above does not need it.
nvcc = shutil.which("nvcc")
if nvcc is None:
    print("nvcc not found. Optional CUDA C++ append test skipped.")
else:
    subprocess.run([PY, "-m", "pip", "install", "-e", ".[cuda-append]"], cwd=REPO, check=True)
    env = os.environ.copy()
    env["KOTFLASH_TEST_CUDA_APPEND"] = "1"
    env["PATH"] = str(Path(PY).parent) + os.pathsep + env.get("PATH", "")
    subprocess.run([PY, "-m", "pytest", "-q", "tests/test_cuda_append.py"], cwd=REPO, env=env, check=True)


## Что сохранять после запуска

Папка `artifacts/t4-validation-*` содержит `pytest.txt`, sweep-результаты для attention длиной 128–2048, decode Q=1/KV=2048 и synthetic decoder-block benchmark. При публикации результатов указывай модель GPU, PyTorch/CUDA/Triton и конкретную форму тензоров. Не смешивай attention-only latency с полной скоростью генерации или обучения модели.